# Supplementary figures


In [ ]:
# Alignment helper is imported from the shared project code.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.transforms import ScaledTranslation
from matplotlib.ticker import FuncFormatter
ROOT=Path.cwd() if (Path.cwd()/'code').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'code'))
from audit_panel_alignment import require_matplotlib_panel_alignment
I=ROOT/'output/Supplementary_Data';F=ROOT/'output/Supplementary_Figures';D=ROOT/'output/Source_Data/Supplementary_Figures';Q=ROOT/'checks/document_sync/figure_qa'
for p in [F,D,Q]:p.mkdir(parents=True,exist_ok=True)
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial'],
 'font.size':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
 'legend.fontsize':7,'svg.fonttype':'none','pdf.fonttype':42,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':0.65,
 'legend.frameon':False,'savefig.dpi':400})
BLUE='#277DA1'; RED='#CB534F'; TEAL='#279C91'; GREY='#6E7781'
def label(ax,letter):
    ax.set_label(letter)
    ax.text(0,1,letter,transform=ax.transAxes+ScaledTranslation(-19/72,9/72,ax.figure.dpi_scale_trans),fontweight='bold',fontsize=10,va='bottom')
def save(fig,n):
    fig.canvas.draw()
    require_matplotlib_panel_alignment(fig,json_out=str(Q/f'Figure_S{n}.alignment.json'),strict=True)
    fig.savefig(F/f'Figure_S{n}.pdf')
    fig.savefig(F/f'Figure_S{n}.svg')
    fig.savefig(F/f'Figure_S{n}.png',dpi=400)
    plt.close(fig)
def read(n):return pd.read_csv(I/(n+'.csv'))


In [ ]:

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(180/25.4,116/25.4));fig.subplots_adjust(left=.015,right=.985,bottom=.03,top=.97)
ax.set(xlim=(0,10),ylim=(0,7));ax.axis('off')
def box(x,y,w,h,text,color='#E9F0F5'):
 ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.08',fc=color,ec='#506070',lw=.7))
 ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=7.5,linespacing=1.4)
def arrow(a,b):ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=10,color='#506070',lw=.8))
box(.25,5.6,2.6,.95,'Roof area and solar resource')
box(3.7,5.6,2.6,.95,'PV installation cohorts\nGeneration and curtailment')
box(7.15,5.6,2.6,.95,'Ambient temperature\nHeatwave exposure')
box(.25,3.35,2.6,1.2,'Regional manufacturing electricity\nPV components, battery packs\nand storage auxiliaries','#E4F2EC')
box(3.7,3.35,2.6,1.2,'Direct use, storage and export\nConversion losses\nDelivered electricity')
box(7.15,3.35,2.6,1.2,'Battery lifetime and additions\nCooling and AC replacement\nRoof replacement difference','#F5EBDD')
box(.25,1.0,2.6,1.2,'PV production burdens\nAllocated to generation\nby production cohort','#E4F2EC')
box(3.7,1.0,2.6,1.2,'Avoided grid emissions\nDeployment-region electricity\nExport absorption assumed')
box(7.15,1.0,2.6,1.2,'Storage production at events\nPack plus auxiliary expansion\nCooling, AC and roof terms','#F5EBDD')
for a,b in [((2.95,6.07),(3.6,6.07)),((5,5.5),(5,4.65)),((8.45,5.5),(8.45,4.65)),((1.55,3.25),(1.55,2.3)),((5,3.25),(5,2.3)),((8.45,3.25),(8.45,2.3))]:arrow(a,b)
ax.text(5,.32,'Net mitigation = avoided grid emissions − PV and storage burdens − cooling and AC terms + roof credit',ha='center',va='center',fontsize=7)
save(fig,1)


# Supplementary figures


In [ ]:
a=pd.read_csv(ROOT/'output/Main_Figure_Data/country_endpoints.csv').query('final_coverage_pct == 75').copy()
assert len(a)==44 and (a.cumulative_net_gt<0).sum()==5
assert np.isclose(a.cumulative_net_gt.sum(),120.024540300157)
a=a.sort_values('cumulative_net_gt')
names=a.country_tag.str.replace('_',' ').str.title().replace({'Rest Of World':'Rest of world','Other':'Rest of world','Row':'Rest of world','Uae':'UAE','Usa':'United States','Uk':'United Kingdom'})
a['net_mitigation_mt_co2e']=a.cumulative_net_gt*1000
a[['country_tag','net_mitigation_mt_co2e']].to_csv(D/'figureS3_regional_balances.csv',index=False)
fig,ax=plt.subplots(figsize=(180/25.4,220/25.4));fig.subplots_adjust(left=.27,right=.96,bottom=.09,top=.98)
ax.barh(np.arange(44),a.net_mitigation_mt_co2e,color=np.where(a.net_mitigation_mt_co2e<0,RED,TEAL),height=.66)
ax.set_yticks(np.arange(44),names);ax.tick_params(axis='y',length=0)
ax.set_xscale('symlog',linthresh=1);ax.set_xticks([-100,-10,-1,0,1,10,100,1000,10000,100000])
ax.xaxis.set_major_formatter(FuncFormatter(lambda x,pos:f'{x:g}'))
ax.axvline(0,color=GREY,lw=.7);ax.set_xlabel('Cumulative net mitigation (Mt CO₂e)')
save(fig,3)


# Supplementary figures


In [ ]:
cov=read('france_boundary_coverage');tim=read('france_boundary_time')
groups=[['Baseline','Low manufacturing','High manufacturing'],['Baseline','Earlier storage','Later storage','No storage'],['Baseline','Zero thermal response','Fixed battery life','Production-event PV accounting']]
assert set(sum(groups,[]))==set(cov.case.unique())==set(tim.case.unique())
fig,axes=plt.subplots(3,2,figsize=(180/25.4,198/25.4));fig.subplots_adjust(left=.13,right=.98,bottom=.07,top=.89,hspace=.74,wspace=.36)
colors=[BLUE,TEAL,RED,'#9467A8'];styles=['-','--','-.',':']
for row,cases in enumerate(groups):
 for col,(data,x,y) in enumerate([(cov,'coverage_pct','net_2050_gt'),(tim,'year','cumulative_net_gt')]):
    ax=axes[row,col];label(ax,'abcdef'[row*2+col])
    for k,case in enumerate(cases):
        z=data[data.case==case].sort_values(x)
        ax.plot(z[x],z[y]*1000,color=colors[k],ls=styles[k],lw=1.2,label=case)
    ax.axhline(0,color=GREY,lw=.65,ls=':');ax.set_ylabel('Net mitigation (Mt CO₂e)')
    ax.set_xlabel('Final roof-reference coverage (%)' if col==0 else 'Year')
    ax.set_xticks([0,25,50,75] if col==0 else [2025,2030,2040,2050])
    if col==0:ax.legend(loc='lower left',bbox_to_anchor=(0,1.075),ncol=1,fontsize=6.5,handlelength=2,borderaxespad=0,labelspacing=.2)
cov.rename(columns={'net_2050_gt':'net_mitigation_gt_co2e'}).to_csv(D/'figureS9_coverage.csv',index=False)
tim.rename(columns={'cumulative_net_gt':'net_mitigation_gt_co2e'}).to_csv(D/'figureS9_time.csv',index=False)
save(fig,9)


# Supplementary figures


In [ ]:
a=read('conditional_spearman').sort_values('spearman_rho')
assert len(a)==14 and a.spearman_rho.between(-1,1).all()
names={'battery_lca_kg_per_kwh_capacity':'Battery production emissions','battery_lca_2050_multiplier':'Battery emissions in 2050','manufacturing_path_exponent':'Battery footprint trajectory','battery_life_multiplier':'Battery lifetime','grid_ef_2050_multiplier':'Grid emissions in 2050','grid_ef_path_exponent':'Grid emissions trajectory','displaced_ef_multiplier':'Displaced-electricity factor','cooling_demand_multiplier':'Cooling demand','roof_credit_multiplier':'Roof contribution','storage_onset_pct':'Storage onset coverage','storage_full_need_pct':'Full storage coverage','storage_sizing_multiplier':'Storage capacity','direct_use_lift':'Direct-use increment','curtailment_onset_pct':'Curtailment onset coverage','curtailment_max_fraction':'Maximum curtailment','deployment_path_exponent':'Deployment trajectory'}
fig,ax=plt.subplots(figsize=(180/25.4,120/25.4));fig.subplots_adjust(left=.36,right=.96,top=.97,bottom=.13)
ax.barh(np.arange(14),a.spearman_rho,color=np.where(a.spearman_rho>0,RED,BLUE),height=.65)
ax.set_yticks(np.arange(14),a.parameter.map(names));ax.tick_params(axis='y',length=0)
ax.axvline(0,color=GREY,lw=.7);ax.set_xlim(-1,1);ax.set_xlabel('Spearman correlation with cumulative net emissions')
a.to_csv(D/'figureS7_parameter_associations.csv',index=False)
save(fig,7)


# Supplementary figures


In [ ]:
a=read('grid_storage');assert len(a)==8
names=a.test.str.replace('Displaced EF','Displaced factor',regex=False)
fig,axes=plt.subplots(1,2,figsize=(180/25.4,100/25.4));fig.subplots_adjust(left=.27,right=.97,bottom=.16,top=.91,wspace=.38)
for ax,col,xlab,letter in zip(axes,['net_gt','shortfall_gt'],['Cumulative net mitigation (Gt CO₂e)','Cumulative net emissions (Gt CO₂e)'],'ab'):
 label(ax,letter);y=np.arange(8)
 ax.scatter(a[col],y,c=[BLUE]*5+[TEAL]*3,s=28,zorder=3)
 baseline=a.loc[a.test=='Displaced EF 1',col].iloc[0]
 ax.axvline(baseline,color=GREY,lw=.75,ls='--')
 ax.set_yticks(y,names if letter=='a' else ['']*8);ax.invert_yaxis();ax.tick_params(axis='y',length=0);ax.set_xlabel(xlab)
 ax.set_xlim(left=0)
a[['test','net_gt','shortfall_gt']].rename(columns={'net_gt':'net_mitigation_gt_co2e','shortfall_gt':'net_emissions_gt_co2e'}).to_csv(D/'figureS8_structural_checks.csv',index=False)
save(fig,8)


# Supplementary figures


In [ ]:
a=pd.read_csv(ROOT/'output/Source_Data/CSV/Figure_5c_values.csv')
assert len(a)==6144 and a.groupby('scenario').size().eq(2048).all()
fig,ax=plt.subplots(figsize=(180/25.4,100/25.4));fig.subplots_adjust(left=.12,right=.96,bottom=.17,top=.95)
for factor,color,ls,name in [('Low',TEAL,'--','Low manufacturing'),('Reference',BLUE,'-','Baseline manufacturing'),('High',RED,'-.','High manufacturing')]:
 z=a[a.scenario==factor].screening_gain_gt.sort_values().to_numpy()
 ax.step(z,np.arange(1,len(z)+1)/len(z),where='post',lw=1.5,color=color,ls=ls,label=name)
ax.axvline(0,color=GREY,lw=.7,ls=':');ax.set_ylim(0,1);ax.set_xlim(left=0)
ax.set_xlabel('Paired screening gain at the same manufacturing scenario (Gt CO₂e)');ax.set_ylabel('Cumulative fraction of scenarios')
ax.legend(loc='lower right',ncol=1)
a.to_csv(D/'figureS6_within_factor_gains.csv',index=False)
save(fig,6)


In [ ]:
from matplotlib.lines import Line2D
full=pd.read_csv(ROOT/'output/Main_Figure_Data/time_paths.csv')
tags=['sweden','norway','france','paraguay','switzerland','iceland']
fig,axes=plt.subplots(2,3,figsize=(7.204724,4.5));fig.subplots_adjust(left=.12,right=.97,bottom=.12,top=.85,hspace=.5,wspace=.42)
rows=[]
for ax,tag,letter in zip(axes.flat,tags,'abcdef'):
    label(ax,letter);z=full[full.country_tag==tag].sort_values('year')
    for col,title,color,ls in [('cumulative_net_gt','Full','#0072B2','-'),('cumulative_reduced_net_gt','Reduced','#777777','--')]:
        ax.plot(z.year,z[col]*1000,c=color,ls=ls,lw=1.1)
        ax.scatter([2050],[z[col].iloc[-1]*1000],s=13,facecolors=color if title=='Full' else 'white',edgecolors=color,lw=.75,zorder=4)
        rows.extend(dict(country=tag,year=int(y),model=title,net_mitigation_mt_co2e=v*1000) for y,v in zip(z.year,z[col]))
    ax.set_title(tag.title(),loc='left',pad=8,fontsize=7)
    ax.axhline(0,c=GREY,lw=.65,ls=':');ax.set(xlim=(2024,2052),xticks=[2025,2035,2050],xlabel='Year');ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
fig.text(.03,.48,'Cumulative net mitigation (Mt CO₂e)',rotation=90,ha='center',va='center',rotation_mode='anchor',fontsize=7)
fig.legend(handles=[Line2D([],[],c='#0072B2',label='Full'),Line2D([],[],c='#777777',ls='--',label='Reduced')],loc='upper right',bbox_to_anchor=(.97,.985),ncol=2,fontsize=7)
pd.DataFrame(rows).to_csv(D/'figureS5_time_paths.csv',index=False);save(fig,5)


In [ ]:
# Setup
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import FuncNorm, LinearSegmentedColormap, TwoSlopeNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.collections import LineCollection
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.transforms import Bbox
ROOT=Path.cwd() if (Path.cwd()/'code').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'code'))
from audit_panel_alignment import require_matplotlib_panel_alignment
D=ROOT/'output/Main_Figure_Data'; O=ROOT/'output/Main_Figures'; Q=ROOT/'checks/main_figures'; S=ROOT/'output/Source_Data/CSV'
for p in [O,Q,S]:p.mkdir(parents=True,exist_ok=True)
mpl.rcParams.update({'font.family':['Arial','DejaVu Sans'],'font.sans-serif':['Arial','DejaVu Sans'],'font.size':7,
    'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,'legend.fontsize':6,
    'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
    'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white','legend.frameon':False})
INK='#233F62';POS='#0072B2';NEG='#D55E00';GREY='#87949C';TEAL='#299B8F'
COLORS={'Low':'#D89000','Reference':'#0072B2','High':'#AA3377'}
SCENARIO_MARKERS={'Low':'s','Reference':'o','High':'^'}
TABLES={}
def read(n):return pd.read_csv(D/(n+'.csv'))
def save_data(n,f):
    assert not f.empty and not f.columns.duplicated().any()
    TABLES[n]=f.reset_index(drop=True);TABLES[n].to_csv(S/(n+'.csv'),index=False)
def name(s):return {'uk':'UK','usa':'USA','row':'Rest of world'}.get(s,s.replace('_',' ').title())
def panel(ax,letter):
    ax.set_label(letter)
    ax.annotate(letter,(0,1),xycoords='axes fraction',xytext=(-6,9),textcoords='offset points',fontsize=9,fontweight='bold',color=INK,annotation_clip=False)
def export(fig,n,axes=None,**kwargs):
    fig.canvas.draw()
    if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
    require_matplotlib_panel_alignment(fig,json_out=str(Q/f'Figure_{n}.alignment.json'),strict=True,tolerance_pt=1.5,**kwargs)
    bbox=Bbox.from_extents(0,8/25.4,fig.get_figwidth(),fig.get_figheight()-8/25.4) if n==4 else Bbox.from_extents(0,7/25.4,fig.get_figwidth(),fig.get_figheight()) if n==5 else None
    fig.savefig(O/f'Figure_{n}.pdf',dpi=600,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.svg',dpi=600,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.png',dpi=300,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.tiff',dpi=600,bbox_inches=bbox,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig);print('Exported Figure',n,flush=True)
def show(n):
    if 'get_ipython' in globals():
        from IPython.display import display,Image
        display(Image(filename=str(O/f'Figure_{n}.png')))
ep=read('country_endpoints');high=ep.query('final_coverage_pct==75').set_index('country_tag')
tags=high.query('cumulative_net_gt<0').sort_values('cumulative_net_gt').index.tolist()
O=ROOT/'output/Supplementary_Figures';Q=ROOT/'checks/document_sync/figure_qa';S=ROOT/'output/Source_Data/Supplementary_Figures'
def export(fig,n,axes=None,**kwargs):
    fig.canvas.draw()
    if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
    require_matplotlib_panel_alignment(fig,json_out=str(Q/f'Figure_S{n}.alignment.json'),strict=True,tolerance_pt=1.5,**kwargs)
    for ext in ['pdf','svg','png']:fig.savefig(O/f'Figure_S{n}.{ext}',dpi=400)
    plt.close(fig)

# Constant-grid diagnostic
countries=tags.copy()
fig,b=plt.subplots(figsize=(7.2047244,3.7));fig.subplots_adjust(left=.14,right=.86,bottom=.18,top=.79)
cells=read('diagnostic_cells');pts=cells[cells.main_displayed].copy();trend=read('diagnostic_trend')
sc=b.scatter(pts.grid_ef_2050_g_per_kwh,pts.diagnostic_g_kwh,c=pts.battery_life,cmap='cividis',s=7,alpha=.65,edgecolors='none',rasterized=True)
import matplotlib.patheffects as pe
b.plot(trend.ef_g_kwh,trend['median'],c='#171717',lw=1.2,ls=(0,(4,3)),path_effects=[pe.Stroke(linewidth=2.3,foreground='white'),pe.Normal()],zorder=4)
def wq(v,w):
    order=np.argsort(v);v=np.asarray(v)[order];w=np.asarray(w)[order]
    return v[np.searchsorted(np.cumsum(w),.5*w.sum(),side='left')]
anchors=[];anchor_handles=[]
for tag,mark in zip(countries,['s','^','D','v','P','o']):
    z=cells.query('country_tag==@tag');xx=wq(z.grid_ef_2050_g_per_kwh,z.cumulative_generation_twh);yy=wq(z.diagnostic_g_kwh,z.cumulative_generation_twh)
    b.scatter(xx,yy,marker=mark,s=31,facecolors='white',edgecolors='#171717',linewidths=.9,zorder=5)
    anchor_handles.append(Line2D([],[],ls='none',marker=mark,mfc='white',mec='#171717',ms=4.5,label=name(tag)))
    anchors.append(dict(country=tag,grid_ef_g_co2e_kwh=xx,diagnostic_balance_g_co2e_kwh=yy))
b.set_yscale('symlog',linthresh=15)
b.set(xlim=(0,108),ylim=(min(-110,pts.diagnostic_g_kwh.min()*1.5),max(100,pts.diagnostic_g_kwh.max()*1.2)),xlabel='Constant grid EF (g CO₂e/kWh)',ylabel='Diagnostic balance (g CO₂e/kWh)',xticks=[0,25,50,75,100])
b.yaxis.set_major_formatter(FuncFormatter(lambda v,p:f'{v:g}'));b.axhline(0,c=GREY,lw=.65,ls=':')
tx=trend.ef_g_kwh.to_numpy();ty=trend['median'].to_numpy();crossings=[]
for i in np.flatnonzero(ty[:-1]*ty[1:]<0):
    crossing=tx[i]-ty[i]*(tx[i+1]-tx[i])/(ty[i+1]-ty[i]);b.axvline(crossing,c=GREY,lw=.65,ls=':');crossings.append(crossing)
cax=fig.add_axes([.90,.20,.012,.55]);cb=fig.colorbar(sc,cax=cax);cb.set_label('Battery calendar life (yr)',fontsize=6,labelpad=3);cb.ax.tick_params(labelsize=6)
fig.legend(handles=anchor_handles,loc='upper center',bbox_to_anchor=(.50,.98),ncol=2,fontsize=5.8,columnspacing=.9,labelspacing=.45,handlelength=1.0)
export(fig,2,[b])
save_data('figureS2_diagnostic_cells',pts[['Id','grid_ef_2050_g_per_kwh','diagnostic_g_kwh','battery_life']])
save_data('figureS2_diagnostic_trend',trend)
save_data('figureS2_diagnostic_countries',pd.DataFrame(anchors))
save_data('figureS2_diagnostic_threshold',pd.DataFrame({'grid_ef_g_co2e_kwh':crossings}))


In [ ]:
# Setup
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import FuncNorm, LinearSegmentedColormap, TwoSlopeNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.collections import LineCollection
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.transforms import Bbox
ROOT=Path.cwd() if (Path.cwd()/'code').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'code'))
from audit_panel_alignment import require_matplotlib_panel_alignment
D=ROOT/'output/Main_Figure_Data'; O=ROOT/'output/Main_Figures'; Q=ROOT/'checks/main_figures'; S=ROOT/'output/Source_Data/CSV'
for p in [O,Q,S]:p.mkdir(parents=True,exist_ok=True)
mpl.rcParams.update({'font.family':['Arial','DejaVu Sans'],'font.sans-serif':['Arial','DejaVu Sans'],'font.size':7,
    'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,'legend.fontsize':6,
    'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
    'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white','legend.frameon':False})
INK='#233F62';POS='#0072B2';NEG='#D55E00';GREY='#87949C';TEAL='#299B8F'
COLORS={'Low':'#D89000','Reference':'#0072B2','High':'#AA3377'}
SCENARIO_MARKERS={'Low':'s','Reference':'o','High':'^'}
TABLES={}
def read(n):return pd.read_csv(D/(n+'.csv'))
def save_data(n,f):
    assert not f.empty and not f.columns.duplicated().any()
    TABLES[n]=f.reset_index(drop=True);TABLES[n].to_csv(S/(n+'.csv'),index=False)
def name(s):return {'uk':'UK','usa':'USA','row':'Rest of world'}.get(s,s.replace('_',' ').title())
def panel(ax,letter):
    ax.set_label(letter)
    ax.annotate(letter,(0,1),xycoords='axes fraction',xytext=(-6,9),textcoords='offset points',fontsize=9,fontweight='bold',color=INK,annotation_clip=False)
def export(fig,n,axes=None,**kwargs):
    fig.canvas.draw()
    if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
    require_matplotlib_panel_alignment(fig,json_out=str(Q/f'Figure_{n}.alignment.json'),strict=True,tolerance_pt=1.5,**kwargs)
    bbox=Bbox.from_extents(0,8/25.4,fig.get_figwidth(),fig.get_figheight()-8/25.4) if n==4 else Bbox.from_extents(0,7/25.4,fig.get_figwidth(),fig.get_figheight()) if n==5 else None
    fig.savefig(O/f'Figure_{n}.pdf',dpi=600,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.svg',dpi=600,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.png',dpi=300,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.tiff',dpi=600,bbox_inches=bbox,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig);print('Exported Figure',n,flush=True)
def show(n):
    if 'get_ipython' in globals():
        from IPython.display import display,Image
        display(Image(filename=str(O/f'Figure_{n}.png')))
ep=read('country_endpoints');high=ep.query('final_coverage_pct==75').set_index('country_tag')
tags=high.query('cumulative_net_gt<0').sort_values('cumulative_net_gt').index.tolist()
O=ROOT/'output/Supplementary_Figures';Q=ROOT/'checks/document_sync/figure_qa';S=ROOT/'output/Source_Data/Supplementary_Figures'
def export(fig,n,axes=None,**kwargs):
    fig.canvas.draw()
    if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
    require_matplotlib_panel_alignment(fig,json_out=str(Q/f'Figure_S{n}.alignment.json'),strict=True,tolerance_pt=1.5,**kwargs)
    for ext in ['pdf','svg','png']:fig.savefig(O/f'Figure_S{n}.{ext}',dpi=400)
    plt.close(fig)

# Figure 3
fig=plt.figure(figsize=(7.2047244,3.25));top=fig.add_gridspec(1,3,left=.16,right=.97,bottom=.23,top=.86,wspace=.23);axes=[]
points=[];shown=tags+['france','germany']
for i,cov in enumerate([25,50,75]):
    ax=fig.add_subplot(top[0,i]);ax.set_label('a'+str(i));axes.append(ax);sub=ep.query('final_coverage_pct==@cov').set_index('country_tag').loc[shown];vv=sub.cumulative_net_gt*1000
    for j,(tag,v) in enumerate(vv.items()):
        color=POS if v>=0 else NEG;ax.plot([0,v],[j,j],c=color,lw=1);ax.scatter(v,j,s=19,c=color,marker='o' if v>=0 else 'v',zorder=4)
        ax.annotate(f'{v:.1f}',(v,j),xytext=(-2 if abs(v)<1 else 0,6),textcoords='offset points',ha='left' if v < -20 else 'right' if v>20 or abs(v)<1 else 'center',fontsize=5.6,c=color)
        points.append(dict(country=tag,coverage_pct=cov,net_mitigation_mt_co2e=v))
    ax.set_xscale('symlog',linthresh=1);ax.set(xlim=(-55,1300),xticks=[-10,0,10,100,1000],ylim=(6.8,-.8),yticks=range(7),yticklabels=[name(t) for t in shown] if i==0 else [])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,p:f'{v:g}'));ax.axvline(0,c=GREY,lw=.6,ls=':');ax.axhline(4.35,c='#B8BDC5',lw=.45,ls=':');ax.set_title(f'{cov}%',fontsize=7,pad=8,color=INK)
    ax.tick_params(axis='x',labelsize=5.7)
fig.text(.565,.075,'Cumulative net mitigation in 2050 (Mt CO₂e)',ha='center',fontsize=7)
export(fig,4,axes);save_data('figureS4_coverage',pd.DataFrame(points))
